## Transformation

- Anpassung der Datentpen

- Renaming der Attribute

- Feature Engineering

- Speichern als Parquet

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as numpy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('../data/raw/credit_risk_dataset.csv')

| name                       | dtype_raw | dtype_pcd | description                                                   |
|----------------------------|-----------|-----------|---------------------------------------------------------------|
| person_age                 | int64     |     int64      | The age of the person                                         |
| person_income              | int64     |       int64    | The income of the person                                      |
| person_home_ownership      | object    |     category      | The living situation of the person                            |
| person_emp_length          | float64   |     float64      | The number of years the person has been working               |
| loan_intent                | object    |     category      | The purpose of the lent money                                 |
| loan_amnt                  | int64     |    int64       | The amount of the loan                                        |
| loan_int_rate              | float64   |    float64       | The interest rate of the loan                                 |
| cb_person_default_on_file  | object    | boolean          | If the person has not paid back a loan in the past            |
| cb_person_cred_hist_length | int64     |     int64      | The number of years the person had their loan already         |
| loan_status                | int64     |     boolean      | If the person is paying back the loan (0), or not paying (1)  |

---
### Umwandlung


In [ ]:
df_dtypes_pcd = df.astype({
    'person_home_ownership': 'category',
    'loan_intent': 'category',
    'loan_status': 'boolean',
})

df_dtypes_pcd['cb_person_default_on_file'] = df_dtypes_pcd['cb_person_default_on_file'].map({'Y': True, 'N': False}).astype('boolean')

df_dtypes_pcd.dtypes

In [ ]:
display(df_dtypes_pcd.head(10))

In [ ]:
df_transformed = df_dtypes_pcd.rename(columns={
    'person_age': 'age',
    'person_income': 'income',
    'person_home_ownership': 'home_ownership',
    'person_emp_length': 'employment_years',
    'loan_intent': 'loan_purpose',
    'loan_amnt': 'loan_amount',
    'loan_int_rate': 'interest_rate',
    'cb_person_default_on_file': 'prior_default',
    'cb_person_cred_hist_length': 'credit_history_years',
    'loan_status': 'is_default',
})

df_transformed['home_ownership'] = df_transformed['home_ownership'].cat.rename_categories({
    'MORTGAGE': 'Mortgage',
    'OTHER': 'Other',
    'OWN': 'Own',
    'RENT': 'Rent',
})

df_transformed['loan_purpose'] = df_transformed['loan_purpose'].cat.rename_categories({
    'DEBTCONSOLIDATION': 'Debt consolidation',
    'EDUCATION': 'Education',
    'HOMEIMPROVEMENT': 'Home improvement',
    'MEDICAL': 'Medical',
    'PERSONAL': 'Personal',
    'VENTURE': 'Venture',
})

display(df_transformed.dtypes)
display(df_transformed.head())

---

---
### Feature Engineering


- **loan_to_income**: Kredithöhe im Verhältnis zum Jahreseinkommen. Das stärkste
  Einzelsignal: Ab einem Verhältnis von 0,3 fallen 69 % der Kredite aus, darunter 15 %.
  
- **employment_years_missing**: Markiert Anträge ohne Angabe zur Beschäftigungsdauer.
  Hier fallen 31,5 % aus statt 21,5 %. Durch die spätere Median-Imputation würde
  diese Information sonst verloren gehen.



In [ ]:
# Verhältnis Kredithöhe zu Einkommen (Einkommen ist nie 0, Minimum 4.000)
df_transformed['loan_to_income'] = df_transformed['loan_amount'] / df_transformed['income']

# Fehlende Beschäftigungsdauer als eigenes Merkmal
df_transformed['employment_years_missing'] = df_transformed['employment_years'].isna().astype('boolean')


### Save the Data in Parquet File

In [ ]:
ordner = Path('../data/processed')
ordner.mkdir(parents=True, exist_ok=True)

df_dtypes_pcd.to_parquet(ordner / 'credit_dtypes_pcd.parquet', index=False)    # alte Namen
df_transformed.to_parquet(ordner / 'credit_transformed.parquet', index=False)  # neue Namen